# ==============================================================================
# 📱 CUSTOMER RETENTION INTELLIGENCE: TELCO CHURN VIA RANDOM FOREST CLASSIFIER
# ==============================================================================
### Mathematical Formulation & Theoretical Context:
Random Forest combines $B$ decorrelated decision trees built on bootstrap samples $\mathcal{D}_b$.
At each internal split, it searches across a random subspace of $m = \sqrt{p}$ candidate features.

Final class prediction uses soft probability voting:
$$P(y = 1 \mid \mathbf{x}) = \frac{1}{B} \sum_{b=1}^B P_b(y = 1 \mid \mathbf{x})$$

Ensemble variance shrinks via tree decorrelation:
$$\text{Var}(\text{Ensemble}) = \rho \sigma^2 + \frac{1 - \rho}{B} \sigma^2$$


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Customer Retention Analytics Environment ready.")


✅ STEP 1: Customer Retention Analytics Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET CLEANUP
Loading Kaggle Telco Customer Churn dataset (7,043 customers).


In [2]:
# STEP 2 & 3: LOAD & SANITIZE
data_path = 'data/customer_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv'
df = pd.read_csv(data_path)

# Drop ID
df = df.drop(columns=['customerID'])

# Convert TotalCharges from string to numeric (blank spaces -> NaN)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].str.strip(), errors='coerce')

df.columns = [c.strip().lower() for c in df.columns]
print(f"📊 Dataset Shape: {df.shape[0]:,} subscribers, {df.shape[1]} attributes")
print(f"Churn distribution: {df['churn'].value_counts().to_dict()}")
df.head(3)


📊 Dataset Shape: 7,043 subscribers, 20 attributes
Churn distribution: {'No': 5174, 'Yes': 1869}


,gender,seniorcitizen,partner,dependents,tenure,phoneservice,multiplelines,internetservice,onlinesecurity,onlinebackup,deviceprotection,techsupport,streamingtv,streamingmovies,contract,paperlessbilling,paymentmethod,monthlycharges,totalcharges,churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `churn` (Yes=1, No=0).


In [3]:
# STEP 4 & 5: SEGREGATION & HYGIENE
X = df.drop(columns=['churn'])
y = df['churn'].map({'Yes': 1, 'No': 0})

print(f"Missing attributes:\n{X.isnull().sum()[X.isnull().sum() > 0]}")
print(f"Churn Rate: {y.mean()*100:.2f}%")


Missing attributes:
totalcharges    11
dtype: int64
Churn Rate: 26.54%


## 🔒 STEP 6: STRATIFIED SPLIT
80/20 train/test partition preserving churn incidence.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Customers: {X_train.shape[0]:,}")
print(f"Test Customers : {X_test.shape[0]:,}")


Train Customers: 5,634
Test Customers : 1,409


## ⚙️ STEP 7: PREPROCESSING PIPELINE
Categorical one-hot encoding and median numeric imputation (scale-invariant).


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', SimpleImputer(strategy='median'), num_cols)
])
print(f"Categorical features: {len(cat_cols)} | Numerical features: {len(num_cols)}")


Categorical features: 15 | Numerical features: 4


/tmp/ipykernel_650815/1886259526.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=['object']).columns.tolist()


## ⚠️ STEP 8: THE SINGLE TREE VS ENSEMBLE FOREST DUEL
Demonstrating how a forest of 150 trees crushes a single high-variance decision tree.


In [6]:
# STEP 8: SINGLE TREE VS RANDOM FOREST
pipe_single = Pipeline([
    ('prep', preprocessor),
    ('tree', DecisionTreeClassifier(random_state=42))
])
pipe_single.fit(X_train, y_train)

pipe_rf_base = Pipeline([
    ('prep', preprocessor),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1))
])
pipe_rf_base.fit(X_train, y_train)

roc_single = roc_auc_score(y_test, pipe_single.predict_proba(X_test)[:, 1])
roc_rf = roc_auc_score(y_test, pipe_rf_base.predict_proba(X_test)[:, 1])

print(f"Single Decision Tree ROC-AUC    : {roc_single*100:.2f}%")
print(f"Random Forest Ensemble ROC-AUC : {roc_rf*100:.2f}% (Variance reduced!)")
print(f"Ensemble Gain                  : +{(roc_rf - roc_single)*100:.2f}% ROC-AUC!")


Single Decision Tree ROC-AUC    : 66.88%
Random Forest Ensemble ROC-AUC : 82.44% (Variance reduced!)
Ensemble Gain                  : +15.56% ROC-AUC!


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA CROSS-VALIDATION
Optimizing `n_estimators`, `max_depth`, and `min_samples_leaf` using 5-Fold Stratified CV.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'rf__n_estimators': [100, 150],
    'rf__max_depth': [8, 12, 16],
    'rf__min_samples_leaf': [2, 5],
    'rf__max_features': ['sqrt', 0.5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_rf_base, param_grid=param_grid, cv=cv, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV ROC-AUC: {grid.best_score_*100:.2f}%")


🏆 Best Hyperparameters: {'rf__max_depth': 8, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 5, 'rf__n_estimators': 150}
🎯 Best 5-Fold CV ROC-AUC: 84.67%


## ⚔️ STEP 10: ALGORITHM DUEL (Random Forest vs Single Decision Tree)
Benchmarking performance gains.


In [8]:
# STEP 10: BENCHMARK
y_pred_tree = pipe_single.predict(X_test)
y_pred_rf = best_model.predict(X_test)

print("="*65)
print(f"Single Decision Tree -> Accuracy: {accuracy_score(y_test, y_pred_tree)*100:.2f}% | ROC-AUC: {roc_single*100:.2f}%")
print(f"Random Forest (Tuned)-> Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.2f}% | ROC-AUC: {roc_auc_score(y_test, best_model.predict_proba(X_test)[:, 1])*100:.2f}%")
print("="*65)


Single Decision Tree -> Accuracy: 74.24% | ROC-AUC: 66.88%


Random Forest (Tuned)-> Accuracy: 80.55% | ROC-AUC: 84.54%


## 📊 STEP 11: EVALUATION REPORT & PERMUTATION IMPORTANCE
Evaluating churn classification report and feature drivers.


In [9]:
# STEP 11: EVALUATION REPORT
print("📋 CUSTOMER RETENTION CLASSIFICATION REPORT:")
print(classification_report(y_test, y_pred_rf, target_names=['Retained (Loyal)', 'Churned']))

cm = confusion_matrix(y_test, y_pred_rf)
print(f"Confusion Matrix:\n{cm}")


📋 CUSTOMER RETENTION CLASSIFICATION REPORT:
                  precision    recall  f1-score   support

Retained (Loyal)       0.84      0.91      0.87      1035
         Churned       0.68      0.51      0.58       374

        accuracy                           0.81      1409
       macro avg       0.76      0.71      0.73      1409
    weighted avg       0.79      0.81      0.80      1409

Confusion Matrix:
[[946  89]
 [185 189]]


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & SMOKE TEST
Deploying pipeline and evaluating sub-2ms subscriber audit latency.


In [10]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/telco_churn_random_forest_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_cust = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_churn = loaded_pipe.predict(sample_cust)[0]
churn_prob = loaded_pipe.predict_proba(sample_cust)[0][1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n📱 LIVE SUBSCRIBER CHURN RISK AUDIT:")
print(f"   Churn Risk Status: {'🚨 HIGH CHURN RISK' if pred_churn == 1 else '✅ RETAINED / LOYAL'}")
print(f"   Risk Probability : {churn_prob*100:.2f}%")
print(f"   Inference Latency: {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/telco_churn_random_forest_pipeline.joblib (912,267 bytes)



📱 LIVE SUBSCRIBER CHURN RISK AUDIT:
   Churn Risk Status: ✅ RETAINED / LOYAL
   Risk Probability : 2.56%
   Inference Latency: 167.516 ms (SLA < 2.0ms: CHECK)
